# ConvNeXt SSL → HyMS-Route — pretrain rồi finetune full model
Bước 1: **SimSiam SSL** ConvNeXt trên đúng bộ data (không nhãn, chỉ split train).
Bước 2: nạp ConvNeXt đã SSL (đông cứng) vào **full HyMS-Route** rồi train như thường.

> Đổi bộ data: chỉ sửa biến `DATASET` ở Mục 3. Bật/tắt SSL bằng `RUN_SSL` ở Mục 5.

## 1. Cài đặt

In [ ]:
# Cài các thư viện còn thiếu (Kaggle/Colab cần Internet ON).
# torch/torchvision đã có sẵn trên Kaggle & Colab; phần dưới chỉ bổ sung cái thiếu.
!pip install -q pytorch-metric-learning
import torch; print('CUDA:', torch.cuda.is_available())

## 2. Lấy code (tự clone từ GitHub nếu chưa có)
Chạy được cả khi mở từ repo đã clone (local) lẫn môi trường trống (Colab).

In [ ]:
# ── Lấy code: tự clone repo nếu chưa có (Colab/máy mới); bỏ qua nếu đã có sẵn ──
import os, sys

REPO_URL = 'https://github.com/trong5nhan6/retrieval.git'
REPO_DIR = 'retrieval'      # tên thư mục sau khi clone

if os.path.exists('config.py'):                       # đang ở ngay trong repo
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'config.py')): # đang ở notebooks/ (clone local)
    PROJECT_ROOT = os.path.abspath('..')
else:                                                 # môi trường trống -> clone
    if not os.path.isdir(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    PROJECT_ROOT = os.path.abspath(REPO_DIR)

os.chdir(PROJECT_ROOT)                # cwd = gốc project (để train.py chạy thẳng)
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

## 3. Chọn bộ data + tải từ Drive
**Chỉ cần sửa biến `DATASET`.** Bộ nào có link Drive sẽ tự tải `.zip` về `datasets/` và giải nén.

In [ ]:
from config import HCFG

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHỌN BỘ DATA — chỉ sửa duy nhất dòng này: 'cub' | 'cars' | 'inshop'       ║
DATASET = 'cub'
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Link Google Drive cho từng bộ (.zip). Điền link rồi chọn DATASET tương ứng ─
DATASET_URLS = {
    'cub':    'https://drive.google.com/file/d/1bwP3KNX1Kaj-sPanx32KrCl4_NVM5m2P/view?usp=drive_link',
    'cars':   '',   # TODO: dán link Drive cho Cars196.zip
    'inshop': '',   # TODO: dán link Drive cho In-shop.zip
}

DATASETS_DIR = os.path.join(PROJECT_ROOT, 'datasets')
os.makedirs(DATASETS_DIR, exist_ok=True)

import re, zipfile

def _drive_id(url):
    m = re.search(r'/d/([A-Za-z0-9_-]+)', url) or re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    return m.group(1) if m else None

def fetch_dataset(name):
    """Tải <name>.zip từ Drive vào datasets/ rồi giải nén (bỏ qua nếu chưa có link)."""
    url = DATASET_URLS.get(name, '')
    if not url:
        print(f'[{name}] chưa có link Drive — bỏ qua (hãy điền DATASET_URLS).')
        return
    import gdown
    zip_path = os.path.join(DATASETS_DIR, f'{name}.zip')
    if not os.path.exists(zip_path):
        print(f'[{name}] tải từ Drive...')
        gdown.download(id=_drive_id(url), output=zip_path, quiet=False)
    print(f'[{name}] giải nén -> {DATASETS_DIR}')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATASETS_DIR)
    print(f'[{name}] xong.')

# Tải đúng bộ đang chọn
fetch_dataset(DATASET)

# ── Trỏ HCFG tới datasets/ (đường dẫn tuyệt đối) ─────────────────────────────
HCFG.data_roots = {
    'cub':    os.path.join(DATASETS_DIR, 'CUB_200_2011'),
    'cars':   os.path.join(DATASETS_DIR, 'Cars196'),
    'inshop': os.path.join(DATASETS_DIR, 'In-shop Clothes Retrieval Benchmark'),
}
HCFG.epochs = 60
print('DATASET =', DATASET, '| root =', HCFG.data_roots[DATASET])

## 4. Chỉnh config trực tiếp (tuỳ chọn)
Sửa các tham số trong `OVERRIDES` để ghi đè `config.py` cho lần chạy này. Áp cho **cả** notebook (smoke-test/eval) **lẫn** `train.py` (qua file JSON). Để `OVERRIDES = {}` nếu dùng mặc định.

In [ ]:
# ── CHỈNH CONFIG TẠI ĐÂY: bỏ comment & sửa giá trị muốn đổi (so với config.py) ──
OVERRIDES = {
    # ── Training ──
    # 'epochs':          60,
    # 'frozen_epochs':   5,       # số epoch warmup (đóng băng backbone)
    # 'finetune_blocks': 2,       # số block ViT mở băng ở Stage-2 (0 = giữ đóng băng)
    # 'head_lr':         1e-4,
    # 'backbone_lr':     1e-5,
    # 'weight_decay':    1e-4,
    # 'grad_clip':       1.0,
    # 'eval_every':      5,        # đánh giá mỗi N epoch

    # ── Sampler / batch ──
    # 'classes_per_batch': 20,     # P lớp / batch
    # 'samples_per_class': 6,      # K ảnh / lớp  (batch_size ~= P*K)
    # 'batch_size':        128,

    # ── Loss ──
    # 'lambda_route':      0.5,    # trọng số routing-consistency (0 = tắt)
    # 'temperature':       0.07,
    # 'route_temperature': 0.1,

    # ── Soft MoE / kiến trúc ──
    # 'n_experts':        8,
    # 'slots_per_expert': 4,
    # 'expert_hidden':    512,
    # 'embed_dim':        256,
    # 'route_dim':        64,
    # 'tokens_per_stage': 64,
    # 'cnn_stages':       [0, 1, 2, 3],   # bỏ s1: [1, 2, 3]

    # ── RouteRank (inference) ──
    # 'rr_beta':    0.3,           # trọng số kênh routing khi fuse (0 = tắt RouteRank)
    # 'rr_topk':    10,
    # 'rr_alpha':   3.0,
    # 'rr_reroute': True,

    # ── Ablation switches (mới) ──
    # 'use_vit':  True,           # False -> tắt nhánh ViT
    # 'use_cnn':  True,           # False -> tắt nhánh CNN (ViT-only)
    # 'use_moe':  True,           # False -> bypass SoftMoE; tự tắt route loss + RouteRank

    # ── LR schedule (mới) ──
    # 'lr_schedule':   'cosine',  # 'cosine' | 'constant'
    # 'warmup_epochs': 0,         # số epoch warmup tuyến tính đầu Stage-1

    # ── Chọn loss chính trên z (mới) ──
    # 'loss_type':      'supcon', # 'supcon' | 'triplet' (khớp baseline triploss)
    # 'triplet_margin': 0.1,
    # 'triplet_miner':  True,     # đào semihard triplets

    # ── Embedding fusion (mới) ──
    # 'embed_dim':       512,      # chiều embedding (bỏ bottleneck 256)
    # 'use_cls_skip':    True,     # đường CLS -> embedding (sàn ≈ baseline)
    # 'local_gate_init': 0.0,      # γ khởi tạo nhánh MoE (0 => bắt đầu ≈ CLS)
    # 'bnneck':          True,     # BatchNorm trước L2 (False -> LayerNorm)
}

import json
# 1) áp cho HCFG đang dùng trong notebook (smoke-test & eval thấy ngay)
for k, v in OVERRIDES.items():
    if hasattr(HCFG, k):
        setattr(HCFG, k, v)
    else:
        print('Bỏ qua key không hợp lệ:', k)
# 2) ghi ra file để train.py (tiến trình con `!python`) đọc và áp y hệt
os.makedirs('results', exist_ok=True)
with open('results/_notebook_overrides.json', 'w', encoding='utf-8') as f:
    json.dump(OVERRIDES, f, ensure_ascii=False, indent=2)
print('Đã áp', len(OVERRIDES), 'override:', OVERRIDES or '(dùng mặc định config.py)')

## 5. SSL pretrain ConvNeXt (SimSiam) — học đặc trưng trên chính bộ data
Trong pipeline gốc ConvNeXt **luôn bị đóng băng** ở weights ImageNet (Stage-2 chỉ mở ViT).
Ở đây ta tự-giám sát (self-supervised, **không nhãn**) ConvNeXt bằng **SimSiam** trên
**đúng split train** (giữ giao thức zero-shot — không đụng test), rồi nạp làm feature
extractor đông cứng cho full model.

> Đặt `RUN_SSL = False` để bỏ qua và dùng thẳng ImageNet weights.
> Checkpoint SSL được lưu vào `results/checkpoints/convnext_ssl_{DATASET}.pt` và tự
> nạp vào full model qua biến môi trường `HYMS_CNN_CKPT` (cả smoke-test lẫn `train.py`).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  SSL PRETRAIN ConvNeXt (SimSiam) — cấu hình (tối ưu cho GPU nhỏ / OOM)     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Chống phân mảnh VRAM — phải set TRƯỚC khi CUDA khởi tạo nhiều tensor lớn.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

RUN_SSL        = True      # False -> bỏ qua SSL, dùng thẳng ImageNet weights
SSL_EPOCHS     = 100       # 100–300 cho CUB nhỏ; tăng nếu còn underfit
# ── Bộ điều khiển BỘ NHỚ (chỉnh khi OOM) ─────────────────────────────────────
SSL_BATCH      = 64        # batch THẬT trên GPU. OOM? -> 32 -> 16
SSL_ACCUM      = 4         # gộp gradient: effective batch = SSL_BATCH * SSL_ACCUM (~256)
SSL_IMG        = HCFG.image_size   # OOM nặng? hạ 224 -> 192 -> 160
SSL_CHANNELS_LAST = True   # convnext chạy nhanh + nhẹ hơn ở channels_last
# ─────────────────────────────────────────────────────────────────────────────
SSL_BASE_LR    = 0.05      # SimSiam: lr = base_lr * eff_batch/256 (SGD)
SSL_WD         = 1e-4
SSL_PROJ_DIM   = 2048
SSL_PRED_DIM   = 512
SSL_NUM_WORKERS = 2        # để 0 nếu chạy trên Windows / RAM thấp

SSL_CKPT = os.path.join(PROJECT_ROOT, 'results', 'checkpoints',
                        f'convnext_ssl_{DATASET}.pt')
os.makedirs(os.path.dirname(SSL_CKPT), exist_ok=True)
eff_bs = SSL_BATCH * SSL_ACCUM
print(f'device={dev} | batch_GPU={SSL_BATCH} x accum={SSL_ACCUM} = eff {eff_bs} '
      f'| img={SSL_IMG} | ckpt-> {SSL_CKPT}')

In [ ]:
# ── Dataset 2-view (KHÔNG nhãn) — CHỈ split train để giữ giao thức zero-shot ──
import random
from PIL import Image, ImageFilter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from data.dml_dataset import _PARSERS, _MEAN, _STD

_splits, _ = _PARSERS[DATASET](HCFG.data_roots[DATASET])   # lấy danh sách ảnh train
SSL_PATHS = [p for p, _ in _splits['train']]               # bỏ nhãn — SSL không cần
print(f'[SSL] {len(SSL_PATHS)} ảnh train (KHÔNG dùng test -> tránh leakage).')

class _GaussianBlur:
    def __init__(self, sigma=(0.1, 2.0)): self.sigma = sigma
    def __call__(self, img):
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(*self.sigma)))

# augment mạnh kiểu SimSiam/SimCLR (2 view khác nhau / ảnh)
ssl_aug = transforms.Compose([
    transforms.RandomResizedCrop(SSL_IMG, scale=(0.2, 1.0)),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([_GaussianBlur()], p=0.5),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

class TwoViewDataset(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), self.tf(img)     # hai view ngẫu nhiên của cùng 1 ảnh

ssl_loader = DataLoader(TwoViewDataset(SSL_PATHS, ssl_aug),
                        batch_size=SSL_BATCH, shuffle=True, drop_last=True,
                        num_workers=SSL_NUM_WORKERS, pin_memory=True,
                        persistent_workers=(SSL_NUM_WORKERS > 0))
print('[SSL] batches/epoch =', len(ssl_loader))

In [ ]:
# ── SimSiam trên ConvNeXt (backbone = HCFG.cnn_name) ─────────────────────────
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm

def build_simsiam():
    cnn = getattr(tvm, HCFG.cnn_name)(weights='DEFAULT')   # warm-start ImageNet
    backbone = cnn.features                                # giữ tham chiếu -> cnn.state_dict() tự cập nhật
    with torch.no_grad():                                  # suy ra số chiều feature
        d_feat = backbone(torch.zeros(1, 3, HCFG.image_size, HCFG.image_size)).shape[1]
    proj = nn.Sequential(
        nn.Linear(d_feat, SSL_PROJ_DIM, bias=False), nn.BatchNorm1d(SSL_PROJ_DIM), nn.ReLU(inplace=True),
        nn.Linear(SSL_PROJ_DIM, SSL_PROJ_DIM, bias=False), nn.BatchNorm1d(SSL_PROJ_DIM), nn.ReLU(inplace=True),
        nn.Linear(SSL_PROJ_DIM, SSL_PROJ_DIM, bias=False), nn.BatchNorm1d(SSL_PROJ_DIM, affine=False),
    )
    pred = nn.Sequential(
        nn.Linear(SSL_PROJ_DIM, SSL_PRED_DIM, bias=False), nn.BatchNorm1d(SSL_PRED_DIM), nn.ReLU(inplace=True),
        nn.Linear(SSL_PRED_DIM, SSL_PROJ_DIM),
    )
    return cnn, backbone, proj, pred, d_feat

class SimSiam(nn.Module):
    def __init__(self, backbone, proj, pred):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projector, self.predictor = proj, pred
    def encode(self, x):
        return self.projector(self.pool(self.backbone(x)).flatten(1))
    def forward(self, x1, x2):
        z1, z2 = self.encode(x1), self.encode(x2)
        p1, p2 = self.predictor(z1), self.predictor(z2)
        return p1, p2, z1.detach(), z2.detach()            # stop-grad trên z

def simsiam_loss(p1, p2, z1, z2):
    d = lambda p, z: -F.cosine_similarity(p, z, dim=-1).mean()
    return 0.5 * (d(p1, z2) + d(p2, z1))

In [ ]:
# ── Train SSL (batch nhỏ + gradient accumulation) + lưu + set env ────────────
import math, time, gc
if RUN_SSL:
    cnn_obj, backbone, proj, pred, d_feat = build_simsiam()
    ss = SimSiam(backbone, proj, pred).to(dev)
    if SSL_CHANNELS_LAST and dev == 'cuda':
        ss = ss.to(memory_format=torch.channels_last)
    ss.train()
    eff_bs  = SSL_BATCH * SSL_ACCUM
    init_lr = SSL_BASE_LR * eff_bs / 256.0          # scale theo EFFECTIVE batch
    opt = torch.optim.SGD(ss.parameters(), lr=init_lr, momentum=0.9, weight_decay=SSL_WD)
    use_amp = (dev == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    print(f'[SSL] feat_dim={d_feat} | eff_batch={eff_bs} | init_lr={init_lr:.4f} '
          f'| epochs={SSL_EPOCHS} | amp={use_amp} | channels_last={SSL_CHANNELS_LAST}')
    for ep in range(1, SSL_EPOCHS + 1):
        lr = init_lr * 0.5 * (1 + math.cos(math.pi * (ep - 1) / SSL_EPOCHS))   # cosine -> 0
        for g in opt.param_groups: g['lr'] = lr
        tot, n, t0 = 0.0, 0, time.time()
        opt.zero_grad(set_to_none=True)
        for i, (x1, x2) in enumerate(ssl_loader):
            x1 = x1.to(dev, non_blocking=True); x2 = x2.to(dev, non_blocking=True)
            if SSL_CHANNELS_LAST and dev == 'cuda':
                x1 = x1.to(memory_format=torch.channels_last)
                x2 = x2.to(memory_format=torch.channels_last)
            with torch.cuda.amp.autocast(enabled=use_amp):
                p1, p2, z1, z2 = ss(x1, x2)
                loss = simsiam_loss(p1, p2, z1, z2) / SSL_ACCUM   # chia để gộp grad
            scaler.scale(loss).backward()
            if (i + 1) % SSL_ACCUM == 0:                          # bước optimizer mỗi ACCUM micro-batch
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
            tot += loss.item() * SSL_ACCUM; n += 1
        if ep == 1 or ep % 10 == 0 or ep == SSL_EPOCHS:
            # loss ∈ [-1, 0]; càng gần -1 (cosine-sim cao) càng tốt. ~ -1 quá sớm = nguy cơ collapse.
            print(f'[SSL] ep{ep:3d}/{SSL_EPOCHS} loss={tot/n:+.4f} lr={lr:.4f} ({time.time()-t0:.0f}s)')
    torch.save({'cnn': cnn_obj.state_dict(), 'cnn_name': HCFG.cnn_name,
                'dataset': DATASET, 'ssl_epochs': SSL_EPOCHS}, SSL_CKPT)
    print('[SSL] saved ->', SSL_CKPT)
    # giải phóng VRAM của kernel TRƯỚC khi train.py (subprocess) chạy
    for _v in ['ss', 'cnn_obj', 'backbone', 'proj', 'pred', 'opt', 'scaler']:
        globals().pop(_v, None)
    gc.collect()
    if dev == 'cuda': torch.cuda.empty_cache()
else:
    print('[SSL] RUN_SSL=False -> bỏ qua, dùng ImageNet weights.')

# QUAN TRỌNG: set env -> HybridEncoder (smoke-test + train.py subprocess) tự nạp SSL weights
if RUN_SSL and os.path.exists(SSL_CKPT):
    os.environ['HYMS_CNN_CKPT'] = SSL_CKPT
    print('HYMS_CNN_CKPT =', os.environ['HYMS_CNN_CKPT'])
else:
    os.environ.pop('HYMS_CNN_CKPT', None)
    print('HYMS_CNN_CKPT đã gỡ -> full model dùng ImageNet weights.')

## 6. Smoke-test shape (tải DINOv2 + ConvNeXt)

In [ ]:
from models.hybrid_encoder import HybridEncoder
from models.hyms_route import HyMSRoute
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = HybridEncoder(HCFG.vit_name, HCFG.cnn_name, dev,
                    use_vit=HCFG.use_vit, use_cnn=HCFG.use_cnn); enc.freeze_all()
m = HyMSRoute(enc, HCFG).to(dev)
z, rho, C = m(torch.randn(2, 3, 224, 224).to(dev))
print('z', z.shape,
      '| rho', None if rho is None else tuple(rho.shape),
      '| C',   None if C   is None else tuple(C.shape))
print('head params:', sum(p.numel() for p in m.head_parameters() if p.requires_grad))

## 7. Train full model (tự nạp ConvNeXt đã SSL)

In [ ]:
!python train.py --dataset {DATASET} --seed 42

In [ ]:
# ── Xem biểu đồ đã auto-lưu trong results/ ───────────────────────────────────
from IPython.display import Image, display
run = f"hyms_{DATASET}_seed42"          # train.py dùng prefix 'hyms_' và seed 42
for p in (f'results/plot_test_{run}.png', f'results/plot_train_{run}.png'):
    if os.path.exists(p):
        print(p); display(Image(p))
    else:
        print('(chưa có)', p)

## 8. Eval lại từ checkpoint (base vs RouteRank)

In [ ]:
from data.dml_dataset import get_dml_loaders
from eval.routerank import evaluate_self, evaluate_query_gallery

L = get_dml_loaders(DATASET, HCFG)
ck = torch.load(f'results/checkpoints/best_hyms_{DATASET}_seed42.pt', map_location=dev)
m.load_state_dict(ck['model'])
rk = HCFG.recall_k_for(DATASET)
if DATASET == 'inshop':                       # query/gallery rời nhau
    r = evaluate_query_gallery(m, L['query'], L['gallery'], dev, HCFG, recall_k=rk)
else:                                          # CUB/Cars: self-retrieval trên test
    r = evaluate_self(m, L['test'], dev, HCFG, recall_k=rk)
print('BASE     :', r['base'])
print('RouteRank:', r.get('routerank', '(tắt — use_moe=False)'))

## Ablation (qua config)
`HCFG.cnn_stages=[1,2,3]` bỏ s1 · `HCFG.lambda_route=0` bỏ L_route · `HCFG.rr_beta=0` tắt RouteRank.